In [3]:
# ============================================================
# EN -> FR Seq2Seq Neural Machine Translation with Attention
# ============================================================
#
# Requirements:
#   pip install tensorflow numpy
#
# Tested conceptually with modern TensorFlow / Keras
# ============================================================

import tensorflow as tf
import numpy as np

from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# ============================================================
# 1. Tiny Toy Parallel Corpus
# ============================================================

pairs = [
    ("i love you", "je t'aime"),
    ("how are you", "comment ça va"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("see you soon", "à bientôt"),
    ("where are you", "où es-tu"),
    ("i am fine", "je vais bien"),
    ("what is your name", "comment tu t'appelles"),
    ("nice to meet you", "ravi de te rencontrer"),
    ("good night", "bonne nuit"),
]


# ============================================================
# 2. Tokenization
# ============================================================

def build_tokenizer(texts, num_words=None, oov="<unk>"):
    tokenizer = Tokenizer(
        num_words=num_words,
        oov_token=oov,
        filters=""
    )
    tokenizer.fit_on_texts(texts)
    return tokenizer


# English sentences
en_texts = [en for en, fr in pairs]

# French sentences
fr_texts = [fr for en, fr in pairs]


# Add special tokens
fr_in_texts = [
    "<sos> " + text
    for text in fr_texts
]

fr_out_texts = [
    text + " <eos>"
    for text in fr_texts
]


# Build tokenizers
tok_en = build_tokenizer(en_texts)

tok_fr = build_tokenizer(
    fr_in_texts + fr_out_texts
)


# ============================================================
# 3. Convert Text -> Integer Sequences
# ============================================================

en_seqs = tok_en.texts_to_sequences(en_texts)

fr_in_seqs = tok_fr.texts_to_sequences(fr_in_texts)

fr_out_seqs = tok_fr.texts_to_sequences(fr_out_texts)


# Maximum lengths
max_len_en = max(
    len(seq)
    for seq in en_seqs
)

max_len_fr = max(
    len(seq)
    for seq in fr_out_seqs
)


# Padding
en_seqs = pad_sequences(
    en_seqs,
    maxlen=max_len_en,
    padding="post"
)

fr_in_seqs = pad_sequences(
    fr_in_seqs,
    maxlen=max_len_fr,
    padding="post"
)

fr_out_seqs = pad_sequences(
    fr_out_seqs,
    maxlen=max_len_fr,
    padding="post"
)


# Vocabulary sizes
en_vocab = len(tok_en.word_index) + 1
fr_vocab = len(tok_fr.word_index) + 1


print("English vocabulary:", en_vocab)
print("French vocabulary:", fr_vocab)

print("English max length:", max_len_en)
print("French max length:", max_len_fr)


# ============================================================
# 4. Create Dataset
# ============================================================

dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        (
            en_seqs,
            fr_in_seqs,
            fr_out_seqs
        )
    )
    .shuffle(100)
    .batch(16)
)


# ============================================================
# 5. Model Hyperparameters
# ============================================================

EMB_EN = 64
EMB_FR = 64

RNN_UNITS = 128


# ============================================================
# 6. Encoder
# ============================================================

class Encoder(tf.keras.Model):

    def __init__(
        self,
        vocab,
        emb_dim,
        rnn_units
    ):
        super().__init__()

        # mask_zero=False because we handle
        # padding manually.
        self.emb = layers.Embedding(
            input_dim=vocab,
            output_dim=emb_dim,
            mask_zero=False
        )

        self.rnn = layers.GRU(
            rnn_units,
            return_sequences=True,
            return_state=True
        )


    def call(self, x):

        # x:
        # (B, T)

        x = self.emb(x)

        # x:
        # (B, T, E)

        seq, state = self.rnn(x)

        # seq:
        # (B, T, H)

        # state:
        # (B, H)

        return seq, state


# ============================================================
# 7. Bahdanau Attention
# ============================================================

class BahdanauAttention(tf.keras.layers.Layer):

    def __init__(self, units):

        super().__init__()

        self.W1 = layers.Dense(units)

        self.W2 = layers.Dense(units)

        self.v = layers.Dense(1)


    def call(
        self,
        enc_seq,
        dec_state,
        mask=None
    ):

        # enc_seq:
        # (B, T, H)

        # dec_state:
        # (B, H)

        dec_state_exp = tf.expand_dims(
            dec_state,
            axis=1
        )

        # enc_seq projection
        w1 = self.W1(enc_seq)

        # decoder state projection
        w2 = self.W2(dec_state_exp)

        # Attention score
        score = self.v(
            tf.nn.tanh(
                w1 + w2
            )
        )

        # (B, T, 1)
        score = tf.squeeze(
            score,
            axis=-1
        )

        # Mask padding tokens
        if mask is not None:

            score = tf.where(
                mask,
                score,
                tf.fill(
                    tf.shape(score),
                    tf.constant(-1e9)
                )
            )

        # Attention weights
        attn = tf.nn.softmax(
            score,
            axis=1
        )

        # Add time dimension
        attn_exp = tf.expand_dims(
            attn,
            axis=1
        )

        # Context vector
        context = tf.matmul(
            attn_exp,
            enc_seq
        )

        # (B, 1, H) -> (B, H)
        context = tf.squeeze(
            context,
            axis=1
        )

        return context, attn


# ============================================================
# 8. Decoder
# ============================================================

class Decoder(tf.keras.Model):

    def __init__(
        self,
        vocab,
        emb_dim,
        rnn_units,
        attention
    ):
        super().__init__()

        # Disable automatic masking.
        # We manually mask encoder padding.
        self.emb = layers.Embedding(
            input_dim=vocab,
            output_dim=emb_dim,
            mask_zero=False
        )

        self.rnn = layers.GRU(
            rnn_units,
            return_sequences=True,
            return_state=True
        )

        self.fc = layers.Dense(
            vocab
        )

        self.attn = attention


    def call(
        self,
        y_prev,
        state,
        enc_seq,
        enc_mask
    ):

        # ----------------------------------------------------
        # Target embedding
        # ----------------------------------------------------

        # y_prev:
        # (B,)

        y_emb = self.emb(y_prev)

        # y_emb:
        # (B, E)


        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        context, attn = self.attn(
            enc_seq,
            state,
            enc_mask
        )

        # context:
        # (B, H)


        # ----------------------------------------------------
        # Combine embedding + context
        # ----------------------------------------------------

        rnn_input = tf.concat(
            [
                y_emb,
                context
            ],
            axis=-1
        )

        # (B, E + H)


        # Add time dimension

        rnn_input = tf.expand_dims(
            rnn_input,
            axis=1
        )

        # (B, 1, E + H)


        # ----------------------------------------------------
        # GRU
        # ----------------------------------------------------

        out, state = self.rnn(
            rnn_input,
            initial_state=state
        )

        # out:
        # (B, 1, H)

        # state:
        # (B, H)


        # Remove time dimension

        out = tf.squeeze(
            out,
            axis=1
        )

        # (B, H)


        # ----------------------------------------------------
        # Vocabulary prediction
        # ----------------------------------------------------

        logits = self.fc(out)

        # (B, vocab)

        return logits, state, attn


# ============================================================
# 9. Create Model
# ============================================================

encoder = Encoder(
    en_vocab,
    EMB_EN,
    RNN_UNITS
)

attention = BahdanauAttention(
    RNN_UNITS
)

decoder = Decoder(
    fr_vocab,
    EMB_FR,
    RNN_UNITS,
    attention
)


# ============================================================
# 10. Loss Function + Optimizer
# ============================================================

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.003
)


# ============================================================
# 11. Training Step
# ============================================================

@tf.function
def train_step(
    en_batch,
    fr_in_batch,
    fr_out_batch
):

    with tf.GradientTape() as tape:

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        enc_seq, enc_state = encoder(
            en_batch
        )

        # Encoder padding mask
        enc_mask = tf.not_equal(
            en_batch,
            0
        )

        # Initial decoder state
        dec_state = enc_state


        # ----------------------------------------------------
        # Teacher forcing
        # ----------------------------------------------------

        total_loss = tf.constant(
            0.0,
            dtype=tf.float32
        )

        total_tokens = tf.constant(
            0.0,
            dtype=tf.float32
        )


        # Number of target tokens
        target_len = tf.shape(
            fr_out_batch
        )[1]


        # Start with <sos>
        y_prev = fr_in_batch[:, 0]


        # Predict each target token
        for t in tf.range(target_len):

            # Decoder prediction
            logits, dec_state, _ = decoder(
                y_prev,
                dec_state,
                enc_seq,
                enc_mask
            )


            # True target token
            y_true = fr_out_batch[:, t]


            # Calculate loss
            step_loss = loss_object(
                y_true,
                logits
            )


            # Ignore padding
            token_mask = tf.cast(
                tf.not_equal(
                    y_true,
                    0
                ),
                tf.float32
            )


            # Apply mask
            step_loss *= token_mask


            # Accumulate
            total_loss += tf.reduce_sum(
                step_loss
            )

            total_tokens += tf.reduce_sum(
                token_mask
            )


            # ------------------------------------------------
            # Teacher forcing
            # ------------------------------------------------

            # During training, give the decoder
            # the correct previous target token.

            if t + 1 < target_len:

                y_prev = fr_in_batch[:, t + 1]


        # Average loss over actual tokens

        batch_loss = (
            total_loss /
            (
                total_tokens +
                1e-9
            )
        )


    # --------------------------------------------------------
    # Gradients
    # --------------------------------------------------------

    variables = (
        encoder.trainable_variables +
        attention.trainable_variables +
        decoder.trainable_variables
    )


    gradients = tape.gradient(
        batch_loss,
        variables
    )


    # Prevent exploding gradients
    gradients, _ = tf.clip_by_global_norm(
        gradients,
        5.0
    )


    optimizer.apply_gradients(
        zip(
            gradients,
            variables
        )
    )


    return batch_loss


# ============================================================
# 12. Train Model
# ============================================================

EPOCHS = 300

print("\nStarting training...\n")


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_loss = 0.0
    batches = 0


    for (
        en_batch,
        fr_in_batch,
        fr_out_batch
    ) in dataset:

        loss = train_step(
            en_batch,
            fr_in_batch,
            fr_out_batch
        )

        epoch_loss += float(
            loss.numpy()
        )

        batches += 1


    avg_loss = (
        epoch_loss /
        batches
    )


    if (
        epoch == 1
        or epoch % 10 == 0
    ):

        print(
            f"Epoch {epoch:03d} "
            f"- Loss: {avg_loss:.4f}"
        )


print("\nTraining completed!")


# ============================================================
# 13. Prepare Inference Dictionaries
# ============================================================

idx2fr = {
    index: word
    for word, index
    in tok_fr.word_index.items()
}


sos_id = tok_fr.word_index.get(
    "<sos>"
)

eos_id = tok_fr.word_index.get(
    "<eos>"
)


print("\nSpecial tokens:")
print("<sos> =", sos_id)
print("<eos> =", eos_id)


# ============================================================
# 14. Translation Function
# ============================================================

def translate(
    sentence_en,
    max_len=20
):

    # --------------------------------------------------------
    # Tokenize English input
    # --------------------------------------------------------

    sequence = tok_en.texts_to_sequences(
        [sentence_en]
    )


    # Padding
    sequence = pad_sequences(
        sequence,
        maxlen=max_len_en,
        padding="post"
    )


    # Convert to Tensor
    sequence = tf.convert_to_tensor(
        sequence,
        dtype=tf.int32
    )


    # --------------------------------------------------------
    # Encoder
    # --------------------------------------------------------

    enc_seq, enc_state = encoder(
        sequence
    )


    # Encoder mask
    enc_mask = tf.not_equal(
        sequence,
        0
    )


    # Initial decoder state
    dec_state = enc_state


    # Start with <sos>
    y_prev = tf.constant(
        [sos_id],
        dtype=tf.int32
    )


    output_tokens = []


    # --------------------------------------------------------
    # Greedy decoding
    # --------------------------------------------------------

    for _ in range(max_len):

        logits, dec_state, attn = decoder(
            y_prev,
            dec_state,
            enc_seq,
            enc_mask
        )


        # Select highest probability token
        next_id = int(
            tf.argmax(
                logits[0]
            ).numpy()
        )


        # Stop at <eos>
        if next_id == eos_id:
            break


        # Ignore padding
        if next_id == 0:
            break


        # Convert ID -> French word
        word = idx2fr.get(
            next_id,
            "<unk>"
        )


        # Don't output special tokens
        if word not in [
            "<sos>",
            "<eos>"
        ]:

            output_tokens.append(
                word
            )


        # Next decoder input
        y_prev = tf.constant(
            [next_id],
            dtype=tf.int32
        )


    return " ".join(
        output_tokens
    )


# ============================================================
# 15. Test Translation
# ============================================================

tests = [
    "i love you",
    "thank you",
    "good night",
    "how are you",
    "where are you",
    "good morning",
    "see you soon",
    "i am fine",
    "what is your name",
    "nice to meet you"
]


print("\n" + "=" * 60)
print("TRANSLATION RESULTS")
print("=" * 60)


for sentence in tests:

    translation = translate(
        sentence
    )

    print(
        f"EN: {sentence:<25} -> FR: {translation}"
    )

English vocabulary: 23
French vocabulary: 25
English max length: 4
French max length: 5

Starting training...

Epoch 001 - Loss: 3.2207
Epoch 010 - Loss: 2.4515
Epoch 020 - Loss: 1.7295
Epoch 030 - Loss: 0.5396
Epoch 040 - Loss: 0.0994
Epoch 050 - Loss: 0.0177
Epoch 060 - Loss: 0.0063
Epoch 070 - Loss: 0.0032
Epoch 080 - Loss: 0.0022
Epoch 090 - Loss: 0.0017
Epoch 100 - Loss: 0.0015
Epoch 110 - Loss: 0.0013
Epoch 120 - Loss: 0.0011
Epoch 130 - Loss: 0.0010
Epoch 140 - Loss: 0.0009
Epoch 150 - Loss: 0.0008
Epoch 160 - Loss: 0.0008
Epoch 170 - Loss: 0.0007
Epoch 180 - Loss: 0.0007
Epoch 190 - Loss: 0.0006
Epoch 200 - Loss: 0.0006
Epoch 210 - Loss: 0.0005
Epoch 220 - Loss: 0.0005
Epoch 230 - Loss: 0.0005
Epoch 240 - Loss: 0.0004
Epoch 250 - Loss: 0.0004
Epoch 260 - Loss: 0.0004
Epoch 270 - Loss: 0.0004
Epoch 280 - Loss: 0.0004
Epoch 290 - Loss: 0.0003
Epoch 300 - Loss: 0.0003

Training completed!

Special tokens:
<sos> = 2
<eos> = 3

TRANSLATION RESULTS
EN: i love you                -> FR